In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
 
# Enter folder here
folder = Path(r"/home/ae19663/Desktop/tekscope_files/")
 
#enter range of folders to analyze, starts from tekscope_20260602_122656, where the date and time are in the format YYYYMMDD_HHMMSS
#now we will analyze all folders in the range from tekscope_20260602_122656 to tekscope_20260602_123000, which corresponds to the date 2026-06-02 and the time range from 12:26:56 to 12:30:00
 
def get_folder_range(start_folder, end_folder):
    all_folders = sorted(folder.glob("tekscope_*"))
    start_index = None
    end_index = None
 
    for i, f in enumerate(all_folders):
        if f.name == start_folder:
            start_index = i
        if f.name == end_folder:
            end_index = i
 
    if start_index is None or end_index is None:
        raise ValueError("Start or end folder not found")
 
    return all_folders[start_index:end_index + 1]
 
print("Enter start folder (e.g. tekscope_20260602_122656):")
start_folder = "tekscope_20260602_161133"
print("Enter end folder (e.g. tekscope_20260602_123000):")
end_folder = "tekscope_20260602_161500"
folders_to_analyze = get_folder_range(start_folder, end_folder)
print(f"Analyzing folders: {[f.name for f in folders_to_analyze]}")
 
BASE_DIR = Path(r"D:\ScopeData")  # parent directory containing tekscope folders
 
 
#will give you folder name go and read csv files and their data columns and plot them
def analyze_folder(folder):
    print(f"Analyzing folder: {folder.name}")
    csv_files = sorted(folder.glob("*.csv"))
    for csv_file in csv_files:
        print(f"Reading file: {csv_file.name}")
        df = pd.read_csv(csv_file, skiprows=10)  # Skip the first row which contains metadata
        print(f"Columns in {csv_file.name}: {df.columns.tolist()}")
        # Plotting the data
        plt.figure(figsize=(10, 6))
        for column in df.columns[1:]:  # Skip the first column (time)
            plt.plot(df[df.columns[0]], df[column], label=column)  # Time vs each channel
        plt.title(f"Data from {csv_file.name}")
        plt.xlabel("Time")
        plt.ylabel("Voltage")
        plt.legend()
        plt.grid()
        #plt.show()
 
 
def plot_csv_together(folder):
    channels_data=[]
    TIME_data=[]
    print(f"Plotting all CSV files in folder: {folder.name}")
    csv_files = sorted(folder.glob("*.csv"))
    for csv_file in csv_files:
        print(f"Reading file: {csv_file.name}")
        df = pd.read_csv(csv_file, skiprows=10)  # Skip the first row which contains metadata
        #append the data of csv files to channels_data and TIME_data
        TIME_data.append(df[df.columns[0]])  # Append time data
        channels_data.append(df[df.columns[1:]])  # Append channel data (all columns except time
    return channels_data, TIME_data
 
#find the rising edge of channels in the channels_data and plot them together with the time data
def find_rising_edge(channels_data, TIME_data):
    rising_edges = []
    for channel in channels_data:
        for col in channel.columns:
            data = channel[col]
            time = TIME_data[0]  # Assuming all time data is the same for each channel
            # Find the rising edge (where the signal goes from low to high)
            for i in range(1, len(data)):
                if data[i-1] < 0.5 and data[i] >= 0.5:  # Threshold of 0.5 for rising edge
                    rising_edges.append((time[i], col))  # Append the time and channel name of the rising edge
                    break  # Stop after finding the first rising edge for this channel
    return rising_edges
 
#now plot the rising edges on the same plot as the original data
def plot_with_rising_edges(folder, channels_data, TIME_data, rising_edges):
    print(f"Plotting data with rising edges for folder: {folder.name}")
    plt.figure(figsize=(12, 8))
    for channel in channels_data:
        for col in channel.columns:
            plt.plot(TIME_data[0], channel[col], label=col)  # Time vs each channel
    for time, channel in rising_edges:
        plt.axvline(x=time, color='r', linestyle='--', label=f'Rising Edge - {channel}')  # Add vertical line for rising edge
    plt.title(f"Data with Rising Edges from {folder.name}")
    plt.xlabel("Time")
    plt.ylabel("Voltage")
    plt.legend()
    plt.grid()
    plt.show()
 
#diffrence between rising edges of different channels
def calculate_rising_edge_differences(rising_edges):
    differences = []
    for i in range(len(rising_edges)):
        for j in range(i + 1, len(rising_edges)):
            time_diff = abs(rising_edges[i][0] - rising_edges[j][0])  # Absolute time difference
            channel_pair = (rising_edges[i][1], rising_edges[j][1])  # Channel names
            differences.append((time_diff, channel_pair))  # Append the time difference and channel pair
    return differences
 
time_differences_all_folders = []
for folder in folders_to_analyze:
    #analyze_folder(folder)
    b = plot_csv_together(folder)
    channels_data = b[0]  # Reset channels data for each folder
    TIME_data = b[1]  # Reset time data for each folder
    rising_edges = find_rising_edge(channels_data, TIME_data)
    plot_with_rising_edges(folder, channels_data, TIME_data, rising_edges)
    rising_edge_differences = calculate_rising_edge_differences(rising_edges)
    time_differences_all_folders.extend(rising_edge_differences)
    print(f"Rising edge time differences for folder {folder.name}:")
    for time_diff, channel_pair in rising_edge_differences:
        print(f"Channels: {channel_pair}, Time Difference: {time_diff}")
 
print ("All analysis complete. Time differences between rising edges across all folders:")
for time_diff, channel_pair in time_differences_all_folders:
    print(f"Channels: {channel_pair}, Time Difference: {time_diff}")
 
            #find the mean and standard deviation of the time differences between rising edges across all folders
time_diff_array = [diff[0] for diff in time_differences_all_folders]
mean_time_diff = np.mean(time_diff_array)
std_time_diff = np.std(time_diff_array)
print(f"Mean Time Difference: {mean_time_diff}")
print(f"Standard Deviation of Time Differences: {std_time_diff}")
 
#only find mean of the time differences between rising edges for a specific channel pair, for example (Channel 1, Channel 2)
specific_channel_pair = ("CH1", "CH2")
specific_time_diffs = [diff[0] for diff in time_differences_all_folders if diff[1] == specific_channel_pair]
mean_specific_time_diff = np.mean(specific_time_diffs)
print(f"Mean Time Difference for {specific_channel_pair}: {mean_specific_time_diff}")
 
#plot thses rising edege time differences in a histogram
plt.figure(figsize=(10, 6))
plt.hist(time_diff_array, bins=20, color='blue', alpha=0.7)
plt.title("Histogram of Time Differences Between Rising Edges")
plt.xlabel("Time Difference (s)")
plt.ylabel("Frequency")
plt.grid()
plt.show()
 